In [ ]:
#Extract Text from Your PDF
import pdfplumber

def extract_text_from_pdf(pdf_path):
    text_data = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            if text:
                text_data.append(text)
    return "\n".join(text_data)  # Combine text from all pages

# Load the text from your PDF
pdf_path = "/Users/devayushrout/Desktop/MedWaste Guardian/legalpdf/2016.pdf"  # Your uploaded PDF
text = extract_text_from_pdf(pdf_path)

# Print a sample of extracted text
print(text[:1000])  # Preview first 1000 characters


BIO-MEDICAL WASTE MANAGEMENT RULES, 2016 as amended till 2019
[Published in the Gazette of India, Extraordinary, Part II, Section 3, Sub-section (i)]
GOVERNMENT OF INDIA
MINISTRY OF ENVIRONMENT, FOREST AND CLIMATE CHANGE
NOTIFICATION
New Delhi, the 28th March, 2016
G.S.R. 343(E).-Whereas the Bio-Medical Waste (Management and Handling) Rules, 1998 was published
vide notification number S.O. 630 (E) dated the 20th July, 1998, by the Government of India in the
erstwhile Ministry of Environment and Forests, provided a regulatory frame work for management of
bio-medical waste generated in the country;
And whereas, to implement these rules more effectively and to improve the collection,
segregation, processing, treatment and disposal of these bio-medical wastes in an environmentally sound
management thereby, reducing the bio- medical waste generation and its impact on the environment, the
Central Government reviewed the existing rules;
And whereas, in exercise of the powers conferred by sect

In [ ]:
#Since my PDF contains tabular data, so i'll extract tables separately using pdfplumber.
import pandas as pd

def extract_tables_from_pdf(pdf_path):
    tables = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            table = page.extract_table()
            if table:
                df = pd.DataFrame(table[1:], columns=table[0])  # Convert to DataFrame
                tables.append(df)
    return tables

# Extract tables
tables = extract_tables_from_pdf(pdf_path)

# Print a sample table
if tables:
    print("\nExtracted Table 1:\n", tables[0])
else:
    print("No tables found in the PDF")



Extracted Table 1:
   Category                                      Type of Waste  \
0      (1)                                                (2)   
1     None  (a) Human Anatomical\nWaste:\nHuman tissues, o...   
2     None  (b)Animal Anatomical\nWaste :\nExperimental an...   
3     None  (c) Soiled Waste:\nItems contaminated\nwith bl...   
4     None                                     (d) Expired or   

            Type of Bag or\nContainer to be\nused  \
0                                             (3)   
1  Yellow coloured\nnon-chlorinated\nplastic bags   
2                                            None   
3                                                   
4                                 Yellow coloured   

                      Treatment and Disposal options  
0                                                (4)  
1  Incineration or Plasma Pyrolysis or\ndeep burial*  
2                                               None  
3  Incineration or Plasma Pyrolysis or\ndeep buri

In [6]:
import chromadb

# Initialize ChromaDB client
chroma_client = chromadb.PersistentClient(path="./chroma_db")  # Make sure this path is correct
collection = chroma_client.get_or_create_collection(name="medical_waste_laws")

# Check if collection is empty
if not collection.count():
    print("Collection is empty. Storing data now...")

    # Add test data (replace this with your actual extracted text)
    text_data = "Biomedical Waste Rules 2016 state that human anatomical waste must be incinerated."
    
    collection.add(
        ids=["doc_1"],  # Unique ID for this document
        documents=[text_data],  # Store extracted text
        metadatas=[{"type": "text", "source": "Biomedical Waste Rules 2016"}]
    )

print("✅ Data stored successfully in ChromaDB!")


Collection is empty. Storing data now...


/Users/devayushrout/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [06:22<00:00, 218kiB/s]   


✅ Data stored successfully in ChromaDB!


In [7]:
print("Existing collections:", chroma_client.list_collections())
print("Number of stored documents:", collection.count())


Existing collections: ['medical_waste_laws']
Number of stored documents: 1


In [8]:
def retrieve_legal_info(query, top_k=3):
    results = collection.query(
        query_texts=[query],
        n_results=top_k
    )
    return results['documents'] if results else ["No relevant documents found."]

query = "What are the disposal rules for human anatomical waste?"
retrieved_docs = retrieve_legal_info(query)
print("\n🔹 Retrieved Legal Rules:\n", retrieved_docs)


Number of requested results 3 is greater than number of elements in index 1, updating n_results = 1



🔹 Retrieved Legal Rules:
 [['Biomedical Waste Rules 2016 state that human anatomical waste must be incinerated.']]


In [9]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# Load LLaMA 2 model & tokenizer
model_name = "meta-llama/Llama-2-7b-hf"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, device_map="auto")


/Users/devayushrout/Desktop/MedWaste Guardian/venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/meta-llama/Llama-2-7b-hf.
401 Client Error. (Request ID: Root=1-67dc594d-22e19f65088a08995c9a9cfe;5d36fad3-d0b0-40ea-b71d-0e582391771a)

Cannot access gated repo for url https://huggingface.co/meta-llama/Llama-2-7b-hf/resolve/main/config.json.
Access to model meta-llama/Llama-2-7b-hf is restricted. You must have access to it and be authenticated to access it. Please log in.

In [ ]:
import chromadb

# Connect to ChromaDB
chroma_client = chromadb.PersistentClient(path="./chroma_db")
collection = chroma_client.get_collection(name="medical_waste_laws")

# Function to retrieve legal text
def retrieve_legal_info(query, top_k=3):
    results = collection.query(query_texts=[query], n_results=top_k)
    return results['documents'] if results else ["No relevant documents found."]
